
# Sumativa 1 — Informe Técnico de Avance del Proyecto

**Universidad Andrés Bello (UNAB)**  
**Magíster en Ciencia de Datos e Inteligencia Artificial**  
**Asignatura:** Estadística Computacional (MCDI501)  
**Equipo:** Grupo 7  
**Dataset:** Por confirmar  
**Fecha:** Octubre de 2026

---

## 1. Introducción

El presente notebook documenta el desarrollo de la primera evaluación sumativa de la asignatura Estadística Computacional, correspondiente al análisis exploratorio e inferencial de un conjunto de datos.

El propósito es aplicar técnicas estadísticas mediante Python, garantizando la reproducibilidad de los procedimientos, la interpretación de los resultados y la fundamentación de las conclusiones obtenidas.

### Objetivo general

Analizar un conjunto de datos mediante herramientas de estadística descriptiva e inferencial para identificar patrones, explorar relaciones entre variables y obtener conclusiones preliminares fundamentadas en evidencia cuantitativa.

### Objetivos específicos

1. Examinar la estructura, integridad y calidad del conjunto de datos.
2. Aplicar técnicas de estadística descriptiva y análisis exploratorio.
3. Identificar relaciones entre variables mediante análisis bivariado y visualizaciones.
4. Calcular estimaciones puntuales e intervalos de confianza del 95 % para al menos tres variables numéricas.
5. Formular y contrastar al menos dos hipótesis estadísticas pertinentes al problema.
6. Interpretar los resultados, reconocer sus limitaciones y establecer próximos pasos.

### Reproducibilidad

El análisis se desarrollará utilizando Python y bibliotecas especializadas en procesamiento, visualización e inferencia estadística. Se documentarán las decisiones metodológicas y se utilizarán semillas aleatorias fijas cuando corresponda.


In [1]:

# ============================================================
# SUMATIVA 1 - ESTADÍSTICA COMPUTACIONAL (MCDI501)
# Grupo 7 | Universidad Andrés Bello
# 2. Importación de bibliotecas y configuración del entorno
# ============================================================

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Estadística descriptiva e inferencial
from scipy import stats

# Configuración de reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configuración de visualizaciones
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 120

# Configuración de visualización de DataFrames
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

print("Entorno de análisis configurado correctamente.")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")


Entorno de análisis configurado correctamente.
Pandas: 3.0.6
NumPy: 2.5.3



## 3. Preparación y auditoría inicial de los datos

### 3.1. Objetivo

Esta etapa tiene como propósito cargar, examinar y preparar el conjunto de datos **E-Commerce Shipping**, asegurando su integridad, trazabilidad y reproducibilidad antes de aplicar técnicas de estadística descriptiva e inferencial.

Para ello, se implementa una arquitectura modular en Python, separando las responsabilidades de carga, validación y transformación de datos en funciones reutilizables ubicadas en el directorio `src/`.

### 3.2. Procedimiento metodológico

La preparación del conjunto de datos contempla las siguientes actividades:

1. **Carga de datos:** lectura del archivo CSV original mediante Pandas, conservando una copia sin modificaciones.

2. **Validación estructural:** verificación de las dimensiones del conjunto de datos, nombres de columnas, tipos de variables y presencia de los atributos requeridos.

3. **Auditoría de calidad:** identificación y cuantificación de valores faltantes, registros duplicados y posibles inconsistencias. Los hallazgos se documentarán antes de aplicar cualquier corrección.

4. **Generación controlada de valores faltantes:** de acuerdo con las instrucciones del curso, cuando el conjunto de datos no presente valores faltantes en variables numéricas, se implementará un mecanismo MCAR (*Missing Completely At Random*).

   Se propone introducir un 10 % de valores faltantes en las variables `Weight_in_gms` y `Cost_of_the_Product`, utilizando una semilla aleatoria fija (`random_state = 42`) para garantizar la reproducibilidad.

5. **Conservación y trazabilidad:** se mantendrán los datos originales y un registro de los valores eliminados artificialmente, permitiendo evaluar posteriormente los procedimientos de imputación.

6. **Persistencia de resultados:** se almacenará una versión preparada del conjunto de datos en `data/processed/`, sin sobrescribir el archivo original.

### 3.3. Criterios metodológicos

- **Reproducibilidad:** todas las transformaciones aleatorias utilizarán una semilla fija.
- **Integridad:** el conjunto de datos original permanecerá inalterado.
- **Trazabilidad:** las modificaciones podrán identificarse y auditarse.
- **Modularidad:** las operaciones de preparación se implementarán mediante funciones independientes y reutilizables.
- **Consistencia:** la versión preparada se utilizará como referencia para las siguientes etapas del proyecto, aplicando los tratamientos de datos faltantes que correspondan a cada análisis.

### 3.4. Resultado esperado

Al finalizar esta etapa, se dispondrá de un conjunto de datos estructurado, validado y documentado, acompañado de un diagnóstico de calidad y un registro de las transformaciones realizadas.

Esta preparación establecerá una base reproducible para el análisis exploratorio, la estimación de parámetros, la formulación de pruebas de hipótesis y las posteriores fases de modelamiento estadístico.

---


In [1]:
# ============================================================
# 3. PREPARACIÓN Y AUDITORÍA INICIAL — E-COMMERCE SHIPPING
# Notebook: notebooks/sumativa_1.ipynb
# ============================================================
from pathlib import Path
import sys
import pandas as pd

# Detectar la raíz del repositorio desde VS Code o Jupyter.
RAIZ = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src").is_dir() and (p / "requirements.txt").is_file()),
    None,
)
if RAIZ is None:
    raise RuntimeError("Abre VS Code en la carpeta raíz del repositorio.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.carga import cargar_csv, guardar_csv
from src.validacion import validar_esquema, informe_calidad, VARIABLES_NUMERICAS
from src.transformacion import generar_faltantes_mcar

RUTA_ORIGINAL = RAIZ / "data" / "raw" / "customer_analytics.csv"
RUTA_PREPARADO = RAIZ / "data" / "processed" / "ecommerce_mcar.csv"
RUTA_VERDAD = RAIZ / "data" / "processed" / "valores_originales_mcar.csv"

# 1) Carga y revisión del original; nunca se sobrescribe data/raw.
df_original = cargar_csv(RUTA_ORIGINAL)
validar_esquema(df_original)
print(f"Dataset original: {len(df_original):,} filas y {df_original.shape[1]} columnas")
print(f"Duplicados exactos: {df_original.duplicated().sum()}")
print("Auditoría inicial:")
display(informe_calidad(df_original))

# 2) La guía exige crear faltantes MCAR solo si no hay faltantes numéricos.
if df_original[list(VARIABLES_NUMERICAS)].isna().any().any():
    df = df_original.copy(deep=True)
    auditoria_mcar = pd.DataFrame(columns=["ID", "variable", "valor_original"])
    print("Se detectaron faltantes numéricos reales: NO se generan artificiales.")
else:
    df, auditoria_mcar = generar_faltantes_mcar(
        df_original,
        columnas=("Weight_in_gms", "Cost_of_the_Product"),
        proporcion=0.10,
        semilla=42,
    )
    print("Faltantes MCAR generados (10% por variable, semilla 42).")

# 3) Guardar productos derivados para futuras sumativas.
guardar_csv(df, RUTA_PREPARADO)
if not auditoria_mcar.empty:
    guardar_csv(auditoria_mcar, RUTA_VERDAD)

print("Auditoría posterior a la preparación:")
display(informe_calidad(df))
print(f"Datos preparados: {RUTA_PREPARADO.relative_to(RAIZ)}")
if not auditoria_mcar.empty:
    print(f"Valores originales protegidos: {RUTA_VERDAD.relative_to(RAIZ)}")
display(df.head())


Dataset original: 10,999 filas y 12 columnas
Duplicados exactos: 0
Auditoría inicial:


,columna,tipo,nulos,porcentaje_nulos,valores_unicos
0,ID,int64,0,0.0,10999
1,Warehouse_block,str,0,0.0,5
2,Mode_of_Shipment,str,0,0.0,3
3,Customer_care_calls,int64,0,0.0,6
4,Customer_rating,int64,0,0.0,5
5,Cost_of_the_Product,int64,0,0.0,215
6,Prior_purchases,int64,0,0.0,8
7,Product_importance,str,0,0.0,3
8,Gender,str,0,0.0,2
9,Discount_offered,int64,0,0.0,65


Faltantes MCAR generados (10% por variable, semilla 42).
Auditoría posterior a la preparación:


,columna,tipo,nulos,porcentaje_nulos,valores_unicos
0,ID,int64,0,0.0,10999
1,Warehouse_block,str,0,0.0,5
2,Mode_of_Shipment,str,0,0.0,3
3,Customer_care_calls,int64,0,0.0,6
4,Customer_rating,int64,0,0.0,5
5,Cost_of_the_Product,float64,1100,10.0,215
6,Prior_purchases,int64,0,0.0,8
7,Product_importance,str,0,0.0,3
8,Gender,str,0,0.0,2
9,Discount_offered,int64,0,0.0,65


Datos preparados: data\processed\ecommerce_mcar.csv
Valores originales protegidos: data\processed\valores_originales_mcar.csv


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177.0,3,low,F,44,1233.0,1
1,2,F,Flight,4,5,216.0,2,low,M,59,3088.0,1
2,3,A,Flight,2,2,183.0,4,low,M,48,3374.0,1
3,4,B,Flight,3,3,176.0,4,medium,M,10,1177.0,1
4,5,C,Flight,2,2,184.0,3,medium,F,46,2484.0,1
